In [ ]:
# ADNI T1 -> DTI Preprocessing Pipeline (Kaggle, GPU-accelerated)

**Pipeline:** read ADNI zip files (already sorted into CN / MCI / AD folders, uploaded as a Kaggle Dataset) -> extract -> convert DICOM to NIfTI -> auto-detect T1 and DTI series -> generate brain mask (ADNI has no premade mask, unlike HCP) -> denoise (MRtrix3) -> skull-strip (HD-BET, GPU) -> fit diffusion tensor (dipy) -> save FA / MD / tensor, organized by CN / MCI / AD.

**Before running:**
1. Upload your zips as a Kaggle Dataset first: on the right sidebar, **+ Add Data -> Upload -> New Dataset**, and keep them organized in `CN/`, `MCI/`, `AD/` subfolders (or one flat folder - the code below handles both, see Step 6).
2. Notebook settings (right sidebar) -> **Accelerator -> GPU T4 x2** (or P100).
3. Session options -> turn **Internet ON** (required for apt/pip installs).

**Run cells top to bottom, in order.**

In [ ]:
!nvidia-smi

## Step 1 - Install MRtrix3 (via NeuroDebian)

Confirm `dwidenoise -version` prints a version number before moving on.

In [ ]:
import subprocess
codename = subprocess.run(["lsb_release", "-cs"], capture_output=True, text=True).stdout.strip()
print("Ubuntu codename detected:", codename)

!wget -qO- http://neuro.debian.net/lists/{codename}.us-ca.libre | sudo tee /etc/apt/sources.list.d/neurodebian.sources.list
!sudo apt-key adv --recv-keys --keyserver hkps://keyserver.ubuntu.com 0xA5D32F012649A5A9 2>/dev/null || \
 sudo apt-key adv --recv-keys --keyserver hkp://keyserver.ubuntu.com:80 0xA5D32F012649A5A9
!sudo apt-get update -qq
!sudo apt-get install -y -qq mrtrix3
!dwidenoise -version

## Step 2 - Install HD-BET (GPU skull stripper)

Confirm `hd-bet -h` prints help text before moving on.

In [ ]:
!pip install -q git+https://github.com/MIC-DKFZ/HD-BET
!hd-bet -h

## Step 3 - Install DICOM conversion tools and refresh the scientific stack

In [ ]:
!pip install -q dicom2nifti pydicom

## STOP - Restart the session now

Kaggle: use the **Run -> Restart session** option (top menu / kebab menu), same reason as before -
freshly installed packages need a clean interpreter. Then continue from the next cell (don't re-run Steps 1-3).

In [ ]:
import numpy as np
import scipy
import dipy
import nibabel as nib
import dicom2nifti
import pydicom
from importlib.metadata import version, PackageNotFoundError

def safe_version(pkg, module=None):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return getattr(module, "__version__", "unknown") if module else "unknown"

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("dipy:", dipy.__version__)
print("nibabel:", nib.__version__)
print("dicom2nifti:", safe_version("dicom2nifti", dicom2nifti))
print("pydicom:", pydicom.__version__)
print("numpy compute check:", np.zeros(5).sum())

from dipy.segment.mask import median_otsu
test = np.random.rand(10, 10, 10)
_ = median_otsu(test, median_radius=1, numpass=1)
print("\ndipy/numba sanity check passed - environment is ready.")

## Step 4 - Configure output location

Kaggle has no Drive equivalent mid-notebook - everything just goes to `/kaggle/working/`,
which persists when you commit ("Save Version") and is downloadable from the Output pane.

In [ ]:
import os

OUT_DIR = "/kaggle/working/processed_adni"
CATEGORIES = ["CN", "MCI", "AD"]

for cat in CATEGORIES:
    os.makedirs(os.path.join(OUT_DIR, cat), exist_ok=True)

WORK_DIR = "/kaggle/working/adni_work_tmp"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Outputs will be saved to: {OUT_DIR}")
print(f"Categories: {CATEGORIES}")

## Step 5 - Helper functions

Unchanged from the Colab version - none of this depended on Colab.

In [ ]:
import subprocess, time, shutil, re, zipfile, glob

# Keywords used to identify T1 vs DTI series from their DICOM SeriesDescription.
# Matched case-insensitively as substrings. Order = priority if multiple matches exist.
T1_KEYWORDS = [
    "mprage", "ir-spgr", "irspgr", "ir-fspgr", "irfspgr", "spgr", "accelerated sag",
]
DTI_KEYWORDS = [
    "dti", "diffusion",
]
# Explicitly exclude these even if they contain a DTI-ish keyword (derived/processed maps, not raw DWI)
DTI_EXCLUDE_KEYWORDS = [
    "fractional", "average dc", "adc", "isotropic", "trace", "fa map", "colormap",
]


def run_cmd(cmd, timeout=None):
    print(f"    $ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result


def find_dicom_series(search_dirs):
    """search_dirs: list of directory paths (all instances of one subject).
    Walks all of them, finds every leaf directory containing .dcm files,
    reads SeriesDescription, and classifies as T1 / DTI / other.
    Returns (t1_dir, dti_dir) - either may be None if not found."""
    import pydicom

    if isinstance(search_dirs, str):
        search_dirs = [search_dirs]

    series_dirs = []
    for base in search_dirs:
        for root, dirs, files in os.walk(base):
            dcm_files = [f for f in files if f.lower().endswith(".dcm") or "." not in f]
            if dcm_files:
                series_dirs.append((root, dcm_files))

    t1_dir, dti_dir = None, None
    for root, dcm_files in series_dirs:
        sample_path = os.path.join(root, dcm_files[0])
        try:
            ds = pydicom.dcmread(sample_path, stop_before_pixels=True, force=True)
            desc = str(getattr(ds, "SeriesDescription", "")).lower()
        except Exception:
            continue

        if any(k in desc for k in DTI_EXCLUDE_KEYWORDS):
            continue

        if t1_dir is None and any(k in desc for k in T1_KEYWORDS):
            t1_dir = root
            print(f"    Found T1 series: '{desc}' ({len(dcm_files)} files)")
        elif dti_dir is None and any(k in desc for k in DTI_KEYWORDS) and len(dcm_files) > 10:
            dti_dir = root
            print(f"    Found DTI series: '{desc}' ({len(dcm_files)} files)")

    return t1_dir, dti_dir


def process_subject(subj, instance_paths, category, out_root, work_root, use_gpu=True):
    t_start = time.time()

    subj_out = os.path.join(out_root, category, subj)
    if os.path.exists(os.path.join(subj_out, "FA.nii.gz")):
        print(f"  [{subj}] Already processed, skipping.")
        return subj
    os.makedirs(subj_out, exist_ok=True)
    subj_work = os.path.join(work_root, f"proc_{subj}")
    os.makedirs(subj_work, exist_ok=True)

    print(f"  [{subj}] Scanning DICOM series across {len(instance_paths)} instance(s)...")
    t1_dicom_dir, dti_dicom_dir = find_dicom_series(instance_paths)
    if t1_dicom_dir is None:
        raise RuntimeError(f"No T1 series found for {subj}")
    if dti_dicom_dir is None:
        raise RuntimeError(f"No DTI series found for {subj}")

    print(f"  [{subj}] Step 1/4: Converting T1 DICOM -> NIfTI...")
    t0 = time.time()
    import dicom2nifti
    dicom2nifti.settings.disable_validate_slice_increment()
    dicom2nifti.settings.disable_validate_orthogonal()
    t1_nifti_path = os.path.join(subj_work, "T1_raw.nii.gz")
    dicom2nifti.dicom_series_to_nifti(t1_dicom_dir, t1_nifti_path, reorient_nifti=True)
    print(f"  [{subj}] Step 1/4 done in {time.time()-t0:.1f}s")

    print(f"  [{subj}] Step 2/4: Converting DTI DICOM -> NIfTI...")
    t0 = time.time()
    dti_nifti_path = os.path.join(subj_work, "dwi_raw.nii.gz")
    result = dicom2nifti.dicom_series_to_nifti(dti_dicom_dir, dti_nifti_path, reorient_nifti=True)
    bval_path = result.get("BVAL_FILE")
    bvec_path = result.get("BVEC_FILE")
    dwi_path = result.get("NII_FILE", dti_nifti_path)
    if not bval_path or not os.path.exists(bval_path):
        raise RuntimeError(f"No bval/bvec extracted for {subj}")
    print(f"  [{subj}] Step 2/4 done in {time.time()-t0:.1f}s")

    # Diagnostic: check DWI array size before denoising, to catch abnormally large volumes early
    import nibabel as nib
    _dwi_probe = nib.load(dwi_path)
    print(f"  [{subj}] DWI shape: {_dwi_probe.shape}, dtype: {_dwi_probe.get_data_dtype()}")
    _n_bytes = _dwi_probe.get_fdata(dtype=np.float32).nbytes if False else None  # skip full load, just report header shape

    print(f"  [{subj}] Step 3/4: Generating brain mask (dwi2mask) + denoising...")
    t0 = time.time()
    mask_path = os.path.join(subj_out, "brain_mask.nii.gz")
    run_cmd(f'dwi2mask "{dwi_path}" "{mask_path}" -fslgrad "{bvec_path}" "{bval_path}" -force')
    denoised_path = os.path.join(subj_work, "dwi_denoised.nii.gz")
    run_cmd(f'dwidenoise "{dwi_path}" "{denoised_path}" -nthreads 4 -force')
    print(f"  [{subj}] Step 3/4 done in {time.time()-t0:.1f}s")

    print(f"  [{subj}] Step 4/4: HD-BET skull stripping + tensor fitting...")
    t0 = time.time()
    t1_brain_path = os.path.join(subj_out, "T1_brain.nii.gz")
    device_flag = "cuda" if use_gpu else "cpu"
    try:
        run_cmd(f'hd-bet -i "{t1_nifti_path}" -o "{t1_brain_path}" -device {device_flag} --disable_tta', timeout=600)
    except Exception as e:
        print(f"    HD-BET failed ({e}) - falling back to dipy median_otsu")
        from dipy.segment.mask import median_otsu
        img = nib.load(t1_nifti_path)
        data = img.get_fdata(dtype=np.float32)
        masked, _ = median_otsu(data, median_radius=4, numpass=4)
        nib.save(nib.Nifti1Image(masked, img.affine), t1_brain_path)

    from dipy.io.gradients import read_bvals_bvecs
    from dipy.core.gradients import gradient_table
    from dipy.reconst.dti import TensorModel, fractional_anisotropy, mean_diffusivity

    dwi_img = nib.load(denoised_path)
    dwi_data = dwi_img.get_fdata(dtype=np.float32)
    affine = dwi_img.affine
    bvals, bvecs = read_bvals_bvecs(bval_path, bvec_path)
    gtab = gradient_table(bvals=bvals, bvecs=bvecs)
    mask = nib.load(mask_path).get_fdata().astype(bool)

    tensor_model = TensorModel(gtab)
    tensor_fit = tensor_model.fit(dwi_data, mask=mask)
    fa = np.clip(np.nan_to_num(fractional_anisotropy(tensor_fit.evals)), 0, 1)
    md_map = np.nan_to_num(mean_diffusivity(tensor_fit.evals))

    nib.save(nib.Nifti1Image(fa.astype(np.float32), affine), os.path.join(subj_out, "FA.nii.gz"))
    nib.save(nib.Nifti1Image(md_map.astype(np.float32), affine), os.path.join(subj_out, "MD.nii.gz"))
    nib.save(nib.Nifti1Image(tensor_fit.lower_triangular().astype(np.float32), affine),
              os.path.join(subj_out, "tensor_lower_triangular.nii.gz"))
    print(f"  [{subj}] Step 4/4 done in {time.time()-t0:.1f}s")

    shutil.rmtree(subj_work, ignore_errors=True)
    print(f"  [{subj}] TOTAL TIME: {time.time()-t_start:.1f}s. Saved to {subj_out}")
    return subj

## Step 6 - Locate your uploaded zips and process, one category at a time

This replaces Colab's `files.upload()` widget. Your zips are already sitting on disk under
`/kaggle/input/<your-dataset-name>/` because you added them as a Dataset before running the notebook.

Run the cell below first - it lists what Kaggle sees under `/kaggle/input/` so you can confirm the path,
then set `DATASET_ROOT` to match. It auto-handles two layouts:
- **Nested:** `DATASET_ROOT/CN/*.zip`, `DATASET_ROOT/MCI/*.zip`, `DATASET_ROOT/AD/*.zip`
- **Flat:** all zips directly in `DATASET_ROOT`, filenames containing "CN"/"MCI"/"AD" (case-insensitive)

If your zips are named/organized differently, tell me the pattern and I'll adjust `find_zips_for_category`.

In [ ]:
import re

DATASET_ROOTS = [
    "/kaggle/input/datasets/puravkhincha/adni-mri-dti",
    "/kaggle/input/datasets/puravkhincha/ad-new",
]
SUBJECT_ID_PATTERN = re.compile(r"^\d{3}_S_\d{4,5}$")


def get_category_roots(category):
    """Returns ALL folders named '<CATEGORY>_ADNI' found anywhere under DATASET_ROOT,
    not just the first — so old and new datasets both get picked up."""
    target = f"{category}_ADNI".lower()
    roots = []
    for root, dirs, files in os.walk(DATASET_ROOT):
        for d in dirs:
            if d.lower() == target:
                roots.append(os.path.join(root, d))
    return roots


def resolve_subject_dirs(category):
    """Groups every folder instance by subject ID across ALL matching category
    roots (old dataset + any newly-added ones) and all MRI_DTI* subfolders within each."""
    cat_roots = get_category_roots(category)
    if not cat_roots:
        print(f"  Could not find any '{category}_ADNI' folder under {DATASET_ROOT}")
        return {}
    print(f"  Found {len(cat_roots)} '{category}_ADNI' folder(s): {cat_roots}")

    subject_map = {}
    for cat_root in cat_roots:
        for root, dirs, files in os.walk(cat_root):
            for d in dirs:
                if SUBJECT_ID_PATTERN.match(d):
                    subject_map.setdefault(d, []).append(os.path.join(root, d))

    print(f"  {category}: found {len(subject_map)} unique subject(s) across all instances/datasets")
    return subject_map


def run_category(category, limit=None):
    assert category in CATEGORIES, f"category must be one of {CATEGORIES}"
    subject_map = resolve_subject_dirs(category)
    subject_ids = sorted(subject_map.keys())
    if limit:
        subject_ids = subject_ids[:limit]
    print(f"Processing {len(subject_ids)} subject(s) for category: {category}")

    succeeded, failed = [], []
    for subj_id in subject_ids:
        instance_paths = subject_map[subj_id]
        print(f"\n=== Processing {subj_id} ({category}) - {len(instance_paths)} instance(s) ===")
        try:
            subj = process_subject(subj_id, instance_paths, category, OUT_DIR, WORK_DIR, use_gpu=True)
            succeeded.append(subj)
        except Exception as e:
            print(f"  FAILED: {e}")
            failed.append(subj_id)

    print("\n" + "="*60)
    print(f"{category} batch done. Succeeded: {len(succeeded)}  Failed: {len(failed)}")
    if failed:
        print("Failed:", ", ".join(failed))
    print("="*60)
    return succeeded, failed

In [ ]:
def get_category_roots(category):
    target = f"{category}_ADNI".lower()
    roots = []
    for base in DATASET_ROOTS:
        for root, dirs, files in os.walk(base):
            for d in dirs:
                if d.lower() == target:
                    roots.append(os.path.join(root, d))
    return roots


def resolve_subject_dirs(category):
    cat_roots = get_category_roots(category)
    if not cat_roots:
        print(f"  Could not find any '{category}_ADNI' folder")
        return {}
    print(f"  Found {len(cat_roots)} '{category}_ADNI' folder(s): {cat_roots}")

    subject_map = {}
    for cat_root in cat_roots:
        for root, dirs, files in os.walk(cat_root):
            matched = [d for d in dirs if SUBJECT_ID_PATTERN.match(d)]
            for d in matched:
                subject_map.setdefault(d, []).append(os.path.join(root, d))
            # stop descending into matched subject folders — no need to walk
            # into their DICOM series subfolders just to count subjects
            dirs[:] = [d for d in dirs if d not in matched]

    print(f"  {category}: found {len(subject_map)} unique subject(s)")
    return subject_map

In [ ]:
NEW_DATASET_TAG = "ad-new"   # matches your dataset path: .../puravkhincha/ad-new/...

subject_map = resolve_subject_dirs("AD")

# Keep only subjects whose paths actually come from the new dataset
new_only = {
    subj: paths for subj, paths in subject_map.items()
    if any(NEW_DATASET_TAG in p for p in paths)
}

print(f"Total AD subjects found: {len(subject_map)}")
print(f"From new dataset only: {len(new_only)}")
print("New subject IDs:", sorted(new_only.keys()))

succeeded, failed = [], []
for subj_id in sorted(new_only.keys()):
    instance_paths = new_only[subj_id]
    print(f"\n=== Processing {subj_id} (AD, new dataset) - {len(instance_paths)} instance(s) ===")
    try:
        subj = process_subject(subj_id, instance_paths, "AD", OUT_DIR, WORK_DIR, use_gpu=True)
        succeeded.append(subj)
    except Exception as e:
        print(f"  FAILED: {e}")
        failed.append(subj_id)

ad_succeeded, ad_failed = succeeded, failed
print(f"\nDone. Succeeded: {len(succeeded)}  Failed: {len(failed)}")

### Run for CN

In [ ]:
cn_succeeded, cn_failed = run_category("CN",limit=30)

### Run for MCI

In [ ]:
mci_succeeded, mci_failed = run_category("MCI")

### Run for AD

In [ ]:
ad_succeeded, ad_failed = run_category("AD")

## Step 7 - Quick visual sanity check on one processed subject

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib

check_category = "CN"
all_succeeded = {"CN": cn_succeeded if "cn_succeeded" in dir() else [],
                  "MCI": mci_succeeded if "mci_succeeded" in dir() else [],
                  "AD": ad_succeeded if "ad_succeeded" in dir() else []}

candidates = all_succeeded.get(check_category, [])
if not candidates:
    for cat, subs in all_succeeded.items():
        if subs:
            check_category = cat
            candidates = subs
            break

if not candidates:
    print("No successfully processed subjects yet - run Step 6 for at least one category first.")
else:
    check_subj = candidates[0]
    subj_out = os.path.join(OUT_DIR, check_category, check_subj)

    t1 = nib.load(os.path.join(subj_out, "T1_brain.nii.gz")).get_fdata()
    fa = nib.load(os.path.join(subj_out, "FA.nii.gz")).get_fdata()

    mid_slice = t1.shape[2] // 2

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(t1[:, :, mid_slice].T, cmap="gray", origin="lower")
    axes[0].set_title(f"{check_subj} ({check_category}) - T1 (skull-stripped)")
    axes[0].axis("off")

    axes[1].imshow(fa[:, :, min(mid_slice, fa.shape[2]-1)].T, cmap="hot", origin="lower")
    axes[1].set_title(f"{check_subj} ({check_category}) - FA map")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    print("If the T1 shows a clean brain (no skull) and the FA map shows bright white matter tracts,")
    print("the pipeline worked correctly.")

In [ ]:
fa = nib.load(os.path.join(subj_out, "FA.nii.gz")).get_fdata()
print("FA shape:", fa.shape)
print("FA nonzero voxel count:", (fa > 0.01).sum(), "out of", fa.size)
print("FA max value:", fa.max(), "FA mean (nonzero):", fa[fa>0.01].mean() if (fa>0.01).any() else "N/A")

mid_fa = fa.shape[2] // 2
plt.figure(figsize=(5,5))
plt.imshow(fa[:, :, mid_fa].T, cmap="hot", origin="lower", vmin=0, vmax=1)
plt.title("FA - own mid slice")
plt.colorbar()
plt.show()

## Step 8 - Zip outputs for download

No `files.download()` on Kaggle. Two ways to get your results out:
1. **Quick:** run the cell below to zip everything into `/kaggle/working/`, then use the **Output** pane
   (right sidebar, after the run finishes) to download `processed_adni.zip` directly.
2. **Persistent:** click **Save Version** (top right) with "Save & Run All" - the zip becomes a downloadable
   notebook Output permanently, without needing to keep the session open.

In [ ]:
import shutil
zip_path = "/kaggle/working/processed_adni.zip"
print("Zipping outputs, this may take a minute...")
shutil.make_archive("/kaggle/working/processed_adni", 'zip', OUT_DIR)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Zip created: {zip_path} ({size_mb:.1f} MB)")
print("Download it from the Output pane on the right once this cell finishes,")
print("or click 'Save Version' to persist it as a notebook output.")